# 09a – Fine-tune ClinicalBERT (text-only model)

Fine-tunes ClinicalBERT on the first-24h clinical notes to produce the text-only baseline
and the fine-tuned encoder used for the fusion models.

**Run after notebook 07** — uses hourly_notes_24h.csv and modelling_cohort_sepsis_mortality.csv.

**Produces:** mbert_best.pt (fine-tuned encoder, used by 09b) and preds_text.npz
(text-only test predictions, used by notebook 13).

MIMIC-III data not included (PhysioNet DUA); see README.

In [ ]:
# --- Setup ---
import os

try:
    from config import DATA_DIR
except ImportError:
    DATA_DIR = os.environ.get("ERP_DATA_DIR", "./data")

def data_path(name):
    return os.path.join(DATA_DIR, name)

import os, numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from transformers import AutoTokenizer, AutoModel

torch.manual_seed(42); np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Load notes + labels, build one text per patient

In [4]:
CKPT_DIR = data_path("mbert_ckpt") + os.sep
os.makedirs(CKPT_DIR, exist_ok=True)

notes = pd.read_csv(data_path("hourly_notes_24h.csv"))
cohort = pd.read_csv(data_path("modelling_cohort_sepsis_mortality.csv"))

notes["TEXT_CLEAN"] = notes["TEXT_CLEAN"].fillna("")
# concatenate the 24 hourly note cells into one text per admission
per_stay = (notes.sort_values(["HADM_ID","hour_bin"])
                 .groupby("HADM_ID")["TEXT_CLEAN"]
                 .apply(lambda s: " ".join([t for t in s if t]))
                 .reset_index())

# attach label + ICUSTAY_ID via cohort (HADM_ID <-> ICUSTAY_ID)
key = cohort[["HADM_ID","ICUSTAY_ID","mortality_after_24h"]].drop_duplicates()
data = per_stay.merge(key, on="HADM_ID", how="inner")
data = data[data["TEXT_CLEAN"].str.len() > 0].reset_index(drop=True)  # keep stays that have text
print("stays with text:", len(data), "| positive rate:", round(data["mortality_after_24h"].mean(),3))

stays with text: 9363 | positive rate: 0.205


## 2. Patient-level split (same seed as the signal baseline for fair comparison)

In [5]:
ids = data["ICUSTAY_ID"].values
y = data.set_index("ICUSTAY_ID")["mortality_after_24h"]

train_ids, temp_ids = train_test_split(ids, test_size=0.30, random_state=42, stratify=y.loc[ids])
val_ids, test_ids   = train_test_split(temp_ids, test_size=0.50, random_state=42, stratify=y.loc[temp_ids])
print("train/val/test:", len(train_ids), len(val_ids), len(test_ids))

split_of = {}
for i in train_ids: split_of[i]="train"
for i in val_ids:   split_of[i]="val"
for i in test_ids:  split_of[i]="test"
data["split"] = data["ICUSTAY_ID"].map(split_of)

train/val/test: 6554 1404 1405


## 3. Dataset / tokeniser

In [ ]:
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
MAX_LEN = 256
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class NoteDataset(Dataset):
    def __init__(self, df):
        self.texts = df["TEXT_CLEAN"].tolist()
        self.labels = df["mortality_after_24h"].values.astype("float32")
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = tokenizer(self.texts[i], truncation=True, max_length=MAX_LEN,
                        padding="max_length", return_tensors="pt")
        return (enc["input_ids"].squeeze(0), enc["attention_mask"].squeeze(0),
                torch.tensor(self.labels[i]))

train_ds = NoteDataset(data[data.split=="train"])
val_ds   = NoteDataset(data[data.split=="val"])
test_ds  = NoteDataset(data[data.split=="test"])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=16)
test_loader  = DataLoader(test_ds, batch_size=16)

## 4. Model: ClinicalBERT + classification head (fine-tuned end-to-end)

In [7]:
class BertClassifier(nn.Module):
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(self.bert.config.hidden_size, 1)
    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:,0]         # [CLS] token
        return self.fc(self.drop(cls)).squeeze(1)

import random
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

model = BertClassifier().to(device)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 5. Fine-tune (grad accumulation, checkpoint each epoch, early stopping)



In [ ]:
pos_rate = float(data[data.split=="train"]["mortality_after_24h"].mean())
pos_weight = torch.tensor([(1-pos_rate)/pos_rate], device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

ACCUM = 2          # effective batch = 8*2 = 16
EPOCHS = 3
START_EPOCH = 0
best_val = 0.0

# For a clean reproduction, start with an empty mbert_ckpt directory.
# If last.pt is present, training resumes automatically from the saved checkpoint.
resume = os.path.join(CKPT_DIR, "last.pt")
if os.path.exists(resume):
    ck = torch.load(resume, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["opt"])
    START_EPOCH = ck["epoch"]; best_val = ck["best_val"]
    print(f"resumed from epoch {START_EPOCH}, best_val={best_val:.3f}")

@torch.no_grad()
def eval_loader(loader):
    model.eval(); ys, ps = [], []
    for ids_, mask, yb in loader:
        ids_, mask = ids_.to(device), mask.to(device)
        p = torch.sigmoid(model(ids_, mask)).cpu().numpy()
        ps.append(p); ys.append(yb.numpy())
    yt, yp = np.concatenate(ys), np.concatenate(ps)
    return roc_auc_score(yt,yp), average_precision_score(yt,yp), f1_score(yt,(yp>=0.5).astype(int))

for epoch in range(START_EPOCH, EPOCHS):
    model.train(); optimizer.zero_grad()
    for step,(ids_,mask,yb) in enumerate(train_loader):
        ids_,mask,yb = ids_.to(device),mask.to(device),yb.to(device)
        loss = criterion(model(ids_,mask), yb) / ACCUM
        loss.backward()
        if (step+1) % ACCUM == 0:
            optimizer.step(); optimizer.zero_grad()
    tr_auc,_,_ = eval_loader(train_loader)
    va_auc,va_ap,va_f1 = eval_loader(val_loader)
    print(f"epoch {epoch+1} | TRAIN AUROC {tr_auc:.3f} | VAL AUROC {va_auc:.3f} AUPRC {va_ap:.3f} F1 {va_f1:.3f}")
    # overfitting warning
    if tr_auc - va_auc > 0.15:
        print("  ⚠️ large train-val gap — possible overfitting")
    # save checkpoints
    torch.save({"model":model.state_dict(),"opt":optimizer.state_dict(),
                "epoch":epoch+1,"best_val":max(best_val,va_auc)}, resume)
    if va_auc > best_val:
        best_val = va_auc
        torch.save(model.state_dict(), os.path.join(CKPT_DIR, "mbert_best.pt"))
        print(f"  saved new best (val AUROC {va_auc:.3f})")

## 6. Test-set performance of the fine-tuned text model

In [14]:
model.load_state_dict(torch.load(os.path.join(CKPT_DIR, "mbert_best.pt"), map_location=device))
te_auc, te_ap, te_f1 = eval_loader(test_loader)
print("=== Fine-tuned ClinicalBERT (text-only, test set) ===")
print(f"AUROC: {te_auc:.3f}\nAUPRC: {te_ap:.3f}\nF1   : {te_f1:.3f}")
print("\nBest val AUROC during training:", round(best_val,3))

=== Fine-tuned ClinicalBERT (text-only, test set) ===
AUROC: 0.669
AUPRC: 0.337
F1   : 0.397

Best val AUROC during training: 0.667


In [ ]:
CKPT_DIR = data_path("mbert_ckpt") + os.sep

# load best model (mbert_best.pt is a plain state_dict)
model.load_state_dict(torch.load(os.path.join(CKPT_DIR, "mbert_best.pt"), map_location=device, weights_only=False))
model.eval()


In [11]:
@torch.no_grad()
def predict(loader):
    model.eval(); ys, ps = [], []
    for ids_, mask, yb in loader:
        p = torch.sigmoid(model(ids_.to(device), mask.to(device))).cpu().numpy()
        ps.append(p); ys.append(yb.numpy())
    return np.concatenate(ys), np.concatenate(ps)

yt, yp = predict(test_loader)
test_stay_ids = data[data.split == "test"]["ICUSTAY_ID"].values
np.savez(data_path("preds_text.npz"), stay_ids=test_stay_ids, y_true=yt, y_prob=yp)
print("saved preds_text.npz", yt.shape)

saved preds_text.npz (1405,)
